In [1]:
import time
import gc

# Video Rendering
import imageio
from IPython.display import HTML
from base64 import b64encode

# VLA requirements
from huggingface_hub import snapshot_download
import numpy as np
from PIL import Image
import torch, torchvision
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

/workspace/venvs/libero/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-09 17:05:37.065668: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-09 17:05:37.289225: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-09 17:05:39.656611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may

In [2]:
# from huggingface_hub import snapshot_download

# REPO = "WaterPancake/minivla-libero90-hf"

# snapshot_download(
#     repo_id=REPO,
#     local_dir="minivla-libero90-hf"
# )

# Loading VLA (either miniVLA or openVLA)

In [ ]:
# (repo_id, unnorm_key)
MODELS = {
    "miniVLA": ("WaterPancake/minivla-libero90-hf", "libero_90"),
    "openVLA": ("openvla/openvla-7b-finetuned-libero-10", "libero_10"),
}

PICK = "openVLA"
REPO, unnorm_key = MODELS[PICK]

processor = AutoProcessor.from_pretrained(REPO, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    REPO,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda").eval()
IMG_SIZE = 224

/workspace/venvs/libero/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- configuration_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- modeling_prismatic.py
. Make sure to double-che

# LIBERO

In [ ]:
# LIBERO 
from libero.libero import benchmark, get_libero_path, set_libero_default_path
from libero.libero.envs import OffScreenRenderEnv
from libero.libero.utils import get_libero_path

In [ ]:
benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict['libero_10']()

# Pringing all 10 task instructions
print('LIBERO-10 tasks:')
for i in range(task_suite.n_tasks):
    task = task_suite.get_task(i)
    print(f'  {i}: {task.language}')

# LIBERO env
task = task_suite.get_task(0)
task_bddl = os.path.join(
    get_libero_path('bddl_files'),
    task.problem_folder,
    task.bddl_file
)
env = OffScreenRenderEnv(
    bddl_file_name=task_bddl,
    camera_heights=IMG_SIZE,
    camera_widths=IMG_SIZE,
)
env.seed(0)
obs = env.reset()

# Set to a fixed initial state for reproducibility
init_states = task_suite.get_task_init_states(0)
obs = env.set_init_state(init_states[0])

# Display the initial observation
img = obs['agentview_image'][::-1]  # flip (LIBERO convention)
display(Image.fromarray(img).resize((384, 384)))

print(f'\nObservation keys: {list(obs.keys())}')
print(f'Image shape: {obs["agentview_image"].shape}')

In [ ]:
def center_crop(image: Image.Image) -> Image.Image:
    w, h = image.size
    crop_w, crop_h = int(0.9 * w), int(0.9 * h)
    left = (w - crop_w) // 2
    top = (h - crop_h) // 2
    return image.crop((left, top, left + crop_w, top + crop_h))

In [ ]:
# ----------------------------
# Rollout config
# ----------------------------
TASK_IDS = list(range(10))
N_ROLLOUTS_PER_TASK = 3
MAX_STEPS = 600
SETTLE_STEPS = 12

SAVE_DIR = "/workspace/libero10_rollouts"
VIDEO_DIR = os.path.join(SAVE_DIR, "videos")
ACTION_DIR = os.path.join(SAVE_DIR, "actions")

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(ACTION_DIR, exist_ok=True)


def center_crop_90_percent(image: Image.Image) -> Image.Image:
    """
    OpenVLA LIBERO fine-tunes expect center 90% crop at eval time.
    """
    w, h = image.size
    crop_w, crop_h = int(0.9 * w), int(0.9 * h)
    left = (w - crop_w) // 2
    top = (h - crop_h) // 2
    return image.crop((left, top, left + crop_w, top + crop_h))


def make_env_for_task(task_id: int):
    """
    Build a fresh LIBERO environment for a task.
    Fresh env per task avoids weird simulator state accumulation.
    """
    task = task_suite.get_task(task_id)
    task_bddl = os.path.join(
        get_libero_path("bddl_files"),
        task.problem_folder,
        task.bddl_file,
    )

    env = OffScreenRenderEnv(
        bddl_file_name=task_bddl,
        camera_heights=IMG_SIZE,
        camera_widths=IMG_SIZE,
    )

    return env, task

def run_single_rollout(task_id: int, rollout_id: int) -> bool:
    
    env, task = make_env_for_task(task_id)

    instruction = task.language
    prompt = f"In: What action should the robot take to {instruction.lower()}?\nOut:"

    init_states = task_suite.get_task_init_states(task_id)

    # Use rollout_id to select among provided LIBERO init states.
    init_state_idx = rollout_id % len(init_states)

    env.seed(rollout_id)
    obs = env.reset()
    obs = env.set_init_state(init_states[init_state_idx])

    video_frames = []
    actions = []
    dones = []
    infos = []

    done = False
    success = False
    t0 = time.time()

    # The object like to jump around after initilization...
    for _ in range(SETTLE_STEPS):
        obs, reward, done, info = env.step(np.zeros(env.env.action_dim))

    for t in range(MAX_STEPS):
        img_array = obs["agentview_image"][::-1]
        video_frames.append(img_array.copy())

        image = Image.fromarray(img_array)
        image = center_crop_90_percent(image)

        inputs = processor(prompt, image).to("cuda", dtype=torch.bfloat16)

        with torch.inference_mode():
            action = vla.predict_action(
                **inputs,
                unnorm_key=unnorm_key,
                do_sample=False,
            )

        action = np.asarray(action, dtype=np.float32)

        # Depending on checkpoint/env convention, this may be needed.
        # action[-1] = 2 * action[-1] - 1
        action = np.clip(action, -1.0, 1.0)

        obs, reward, done, info = env.step(action)

        actions.append(action.copy())
        dones.append(bool(done))
        infos.append(info)

        if info.get("success", False):
            success = True
            break
        if done:
            break

        del inputs
        torch.cuda.empty_cache()

    elapsed = time.time() - t0

    # ----------------------------
    # Save files
    # ----------------------------
    base_name = f"task{task_id:02d}_rollout{rollout_id:02d}"

    video_path = os.path.join(VIDEO_DIR, f"{base_name}.mp4")
    actions_path = os.path.join(ACTION_DIR, f"{base_name}_actions.npy")

    imageio.mimsave(video_path, video_frames, fps=20)
    np.save(actions_path, np.asarray(actions, dtype=np.float32))

    # Clean up env/sim resources aggressively, because MuJoCo + notebooks = soup.
    try:
        env.close()
    except Exception:
        pass

    del env
    gc.collect()
    torch.cuda.empty_cache()

    return success


In [ ]:
for task_id in TASK_IDS:
    print(f'Running Task {task_id}: \"{task_suite.get_task(task_id).language}\" ')

    for rollout_id in range(N_ROLLOUTS_PER_TASK):
        print(f"   rollout_id: {rollout_id}: RUNNING", end="\r", flush=True)
        result = run_single_rollout(task_id, rollout_id)
        print(f"   rollout_id: {rollout_id}: {'[SUCCESS]' if result else '[FAILED]'}")

print("Jobs done :^)")
        